# Web Scraping + Regressão Linear para previsão de preço de imóveis

> Neste web scraper podemos verificar preço de imóveis de Brasília, especialmente no **Plano Piloto** e arredores.

**Etapa 1 — Coleta.** Este notebook baixa os anúncios do VivaReal, extrai os campos de cada card e salva tudo **como texto cru** em `imoveis_brutos.csv`. A conversão para números fica no `02_limpeza_eda.ipynb`.

```text
VivaReal ──(curl_cffi + BeautifulSoup)──▶ imoveis_brutos.csv ──▶ 02_limpeza_eda ──▶ 03_modelagem
```

## Por que o `requests` recebia 403?

O bloqueio **não tem nada a ver com o `robots.txt`**. O `robots.txt` é só um aviso ("por favor, robôs, não acessem X"); nenhum servidor o usa para bloquear ninguém. Quem bloqueia é o **Cloudflare**, que fica na frente do VivaReal e decide se a conexão parece um navegador **antes de olhar os headers**.

Ele compara três "impressões digitais" da conexão:

| Camada | O que o Cloudflare olha | `requests` | Chrome real |
|---|---|---|---|
| **TLS** (handshake HTTPS) | Ordem das cifras, extensões, curvas → hash **JA3/JA4** | Assinatura do OpenSSL/urllib3, conhecida como "Python" | Assinatura do BoringSSL do Chrome |
| **HTTP/2** | Frames `SETTINGS`, ordem dos pseudo-headers | `requests` nem fala HTTP/2 (só HTTP/1.1) | HTTP/2 com parâmetros do Chrome |
| **Headers** | `User-Agent`, `Accept`, `Sec-CH-UA`... | O que você escrever | Coerentes com a versão do navegador |

Trocar o `User-Agent` só altera a **última** camada. Quando o TLS diz "sou Python" e o `User-Agent` diz "sou Chrome 91", a incoerência é justamente o sinal que o Cloudflare procura, e a resposta vem com a página *"Sorry, you have been blocked"*.

**Solução:** a biblioteca [`curl_cffi`](https://github.com/lexiforest/curl_cffi). Ela usa um `curl` compilado para reproduzir **exatamente** o handshake TLS e HTTP/2 do Chrome (`impersonate="chrome"`) e tem a mesma API do `requests`, então o resto do código não muda. Não é preciso Selenium/Playwright, porque o HTML já chega pronto do servidor (os `data-cy` dos cards estão no HTML), sem depender de JavaScript.

> ⚖️ **Limites éticos que este coletor respeita:** não resolve CAPTCHA nem desafios JavaScript, usa só URLs permitidas pelo `robots.txt` (caminhos "limpos" + `?pagina=N`, sem `onde=`, `tipos=`, `quartos=` ou `precoMaximo=`), espera 2 a 4 s entre páginas, não coleta nome nem telefone de anunciante (LGPD) e é para uso acadêmico.

## Imports

Se faltar a biblioteca: `pip install curl_cffi` com o `.venv` ativo.

In [1]:
import random
import re
import time
from datetime import date
from pathlib import Path

import pandas as pd
import requests                                   # só para demonstrar o bloqueio (403)
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi            # mesma API do requests, com "impressão digital" do Chrome
from curl_cffi.requests.exceptions import RequestException

## Setando variáveis

Em vez da busca geral de Brasília (que devolve o **DF inteiro**: Samambaia, Ceilândia, Taguatinga...), buscamos **bairro por bairro** pelas URLs "limpas" do site:

```text
https://www.vivareal.com.br/venda/distrito-federal/brasilia/bairros/asa-sul/apartamento_residencial/?pagina=2
                                                            └─ bairro ──┘ └─── tipo ────────────┘
```

Os *slugs* foram descobertos navegando no site (links `bairros/...` dentro do HTML). Slug inexistente devolve **404**, e o coletor simplesmente pula essa combinação.

In [2]:
CIDADE_URL = "https://www.vivareal.com.br/venda/distrito-federal/brasilia/"
CIDADE_NOME = "Brasília"

# nome que vai para a coluna "bairro"  →  pedaço da URL (.../bairros/<slug>/)
BAIRROS = {
    "Asa Sul": "asa-sul",
    "Asa Norte": "asa-norte",
    "Noroeste": "setor-noroeste",
    "Sudoeste": "setor-sudoeste",
    "Octogonal": "area-octogonal",
    "Cruzeiro Novo": "cruzeiro-novo",
    "Lago Sul": "lago-sul",
    "Lago Norte": "lago-norte",
}

# nome que vai para a coluna "tipo"  →  pedaço da URL
# A ORDEM IMPORTA: a busca de "apartamento" também devolve coberturas e kitnets. Como a
# remoção de repetidos fica com a 1ª ocorrência, os tipos específicos vêm antes.
TIPOS = {
    "cobertura": "cobertura_residencial",
    "kitnet": "kitnet_residencial",
    "casa": "casa_residencial",
    "apartamento": "apartamento_residencial",
}

ANUNCIOS_POR_PAGINA = 30
MAX_PAGINAS_POR_BUSCA = 10          # teto de segurança: 10 × 30 = 300 anúncios por bairro×tipo
PAUSA_SEGUNDOS = (2, 4)             # espera aleatória entre páginas
IMPERSONATE = "chrome"              # perfil TLS/HTTP2 que o curl_cffi imita

ARQUIVO_BRUTO = Path("imoveis_brutos.csv")
FORCAR_COLETA = False               # True = raspa de novo mesmo se o CSV já existir

SELETOR_CARD = '[data-cy="rp-property-cd"]'
url = CIDADE_URL + "apartamento_residencial/"    # busca geral, usada só no teste de depuração abaixo

## Melhorando o Output

In [3]:
def exibir_caixa(titulo, linhas):
    # Desenha uma caixa com bordas arredondadas; `linhas` é uma lista de (rótulo, valor)
    conteudos = [f" {chave}: {'Não informado' if valor in (None, '') else valor}" for chave, valor in linhas]
    largura = max(len(titulo) + 2, *(len(c) + 1 for c in conteudos))

    print("╭" + "─" * largura + "╮")
    print("│" + titulo.center(largura) + "│")
    print("├" + "─" * largura + "┤")
    for conteudo in conteudos:
        print("│" + conteudo.ljust(largura) + "│")
    print("╰" + "─" * largura + "╯")


def exibir_imovel(imovel, titulo="RESUMO DO IMÓVEL"):
    # Aqui os valores ainda são texto cru ("R$ 1.200.000", "75 m²"); a conversão é no notebook 02
    exibir_caixa(titulo, [
        ("Bairro (busca)", imovel.get("bairro")),
        ("Bairro (anúncio)", imovel.get("bairro_anuncio")),
        ("Tipo", imovel.get("tipo")),
        ("Preço", imovel.get("preco")),
        ("Condomínio", imovel.get("condominio")),
        ("IPTU", imovel.get("iptu")),
        ("Área", imovel.get("area_m2")),
        ("Quartos", imovel.get("quartos")),
        ("Banheiros", imovel.get("banheiros")),
        ("Vagas", imovel.get("vagas")),
    ])

## Depuração: `requests` × `curl_cffi`

Mesma URL, duas bibliotecas. Com `requests` (mesmo com `User-Agent` de Chrome) vem **403** e a página de bloqueio do Cloudflare. Com `curl_cffi` vem **200** e 30 cards.

> ⚠️ Com `curl_cffi` **não** passe um `User-Agent` próprio: o `impersonate` já manda headers coerentes com a versão do Chrome imitada. Um `User-Agent` de outra versão (como o `Chrome/91` antigo) cria justamente a incoerência que denuncia o robô.
>
> 💡 O body tem ~1,3 milhão de caracteres. Imprimir tudo trava o VS Code, então imprimimos só o `<title>` e o começo.

In [4]:
headers = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"),
    "Accept-Language": "pt-BR,pt;q=0.9",
}


def resumo_resposta(nome, resposta):
    sopa = BeautifulSoup(resposta.text, "html.parser")
    titulo = sopa.title.get_text(strip=True) if sopa.title else "(sem <title>)"
    print(f"[{nome}]")
    print(f"Status: {resposta.status_code}")
    print(f"URL: {resposta.url}")
    print(f"Tamanho do body: {len(resposta.text):,} caracteres")
    print(f"<title>: {titulo}")
    print(f"Cards de anúncio: {len(sopa.select(SELETOR_CARD))}")
    print("Início do body:", " ".join(resposta.text[:200].split()), "...")
    print()


# 1) requests puro: headers de Chrome, mas o handshake TLS denuncia o Python
resposta = requests.get(url, params={"pagina": 1}, headers=headers, timeout=20)
resumo_resposta("requests", resposta)

# 2) curl_cffi: handshake TLS + HTTP/2 idênticos aos de um Chrome real
resposta = cffi.get(url, params={"pagina": 1}, impersonate=IMPERSONATE, timeout=20)
resumo_resposta("curl_cffi", resposta)

[requests]
Status: 403
URL: https://www.vivareal.com.br/venda/distrito-federal/brasilia/apartamento_residencial/?pagina=1
Tamanho do body: 5,487 caracteres
<title>: Attention Required! | Cloudflare
Cards de anúncio: 0
Início do body: <!DOCTYPE html> <!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]--> <!--[if IE 7]> <html class="no-js ie7 oldie" lang="en-US"> <![endif]--> <!--[if IE 8]> <html class="no- ...



[curl_cffi]
Status: 200
URL: https://www.vivareal.com.br/venda/distrito-federal/brasilia/apartamento_residencial/?pagina=1
Tamanho do body: 1,268,411 caracteres
<title>: Apartamentos para venda em Brasília, DF - Viva Real
Cards de anúncio: 30
Início do body: <!DOCTYPE html><html lang="pt-BR"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="image" href="https://resizedimgs.vivareal.co ...



## Baixando páginas com sessão, retry e backoff

- **Sessão** (`cffi.Session`): reaproveita a conexão e guarda os cookies que o Cloudflare devolve (como `__cf_bm`), como um navegador faz.
- **Retry com backoff exponencial**: em `403/429/5xx` ou *timeout*, espera 5 s, 10 s, 20 s... e tenta de novo. Na primeira coleta de teste, disparar muitas páginas seguidas causou um *timeout*, e é esse tipo de falha temporária que o retry absorve.
- **404** significa que aquela combinação bairro × tipo não existe (ex.: apartamento no Lago Sul). Não é erro, é só "pular".

In [5]:
STATUS_TEMPORARIOS = {403, 429, 500, 502, 503, 504}


def criar_sessao():
    sessao = cffi.Session(impersonate=IMPERSONATE)
    sessao.headers.update({"Accept-Language": "pt-BR,pt;q=0.9"})
    return sessao


def pausa_educada():
    time.sleep(random.uniform(*PAUSA_SEGUNDOS))


def baixar_pagina(sessao, url, pagina, tentativas=4):
    # Devolve a "sopa" da página, ou None se a busca não existir (404)
    for tentativa in range(1, tentativas + 1):
        try:
            resposta = sessao.get(url, params={"pagina": pagina}, timeout=30)
        except RequestException as erro:
            motivo = type(erro).__name__
        else:
            if resposta.status_code == 200:
                return BeautifulSoup(resposta.text, "html.parser")
            if resposta.status_code == 404:
                return None
            if resposta.status_code not in STATUS_TEMPORARIOS:
                resposta.raise_for_status()
            motivo = f"HTTP {resposta.status_code}"

        if tentativa < tentativas:
            espera = 5 * 2 ** (tentativa - 1) + random.uniform(0, 2)
            print(f"   ⚠ {motivo} (tentativa {tentativa}/{tentativas}); aguardando {espera:.0f}s...")
            time.sleep(espera)

    raise RuntimeError(f"Falhou após {tentativas} tentativas ({motivo}): {url}?pagina={pagina}")

## Extraindo os dados de cada card

Anatomia de um card (resumida):

```html
<li data-cy="rp-property-cd" [data-type="FIXED DEVELOPMENTS POSITIONS"]>   ← lançamento
  <a href="https://www.vivareal.com.br/imovel/...-id-2913833141/">
  <div data-cy="rp-cardProperty-location-txt">Asa Sul, Brasília</div>
  <div data-cy="rp-cardProperty-price-txt">
    <p>R$ 1.150.000</p>                          ← preço
    <p>Cond. R$ 1.100 • IPTU R$ 3.000</p>        ← condomínio e IPTU
  </div>
  <li data-cy="rp-cardProperty-propertyArea-txt"><span class="sr-only">Tamanho do imóvel</span> 75 m²</li>
  ...
```

- `.sr-only` é texto invisível para leitores de tela. Removemos com `.decompose()`, senão a área vira `"Tamanho do imóvel 75 m²"`.
- **Bairro:** o rótulo que o anunciante escreve é pouco confiável. Uma busca em *Lago Sul* devolve cards com "Setor de Habitações Individuais Sul" ou só "Brasília", e alguns da Asa Norte vêm como "Norte". Por isso a coluna `bairro` recebe o **bairro da busca** (padronizado), e o texto original fica em `bairro_anuncio` para conferência.
- **`id`** vem do link do anúncio e serve para remover repetidos: anúncios "Destaque" reaparecem em várias páginas, e coberturas/kitnets aparecem também na busca de apartamentos. Cards que o próprio site agrupa (mesmo imóvel, várias imobiliárias) vêm com `href="#"` e ficam sem `id`; esses repetidos são tratados no notebook 02.

In [6]:
def texto(card, data_cy):
    # Devolve o texto visível do elemento com aquele data-cy, ou None se não existir
    elemento = card.select_one(f'[data-cy="{data_cy}"]')
    if elemento is None:                     # alguns anúncios não têm, por exemplo, vagas
        return None
    for escondido in elemento.select(".sr-only"):
        escondido.decompose()                # remove o texto para leitor de tela
    return elemento.get_text(" ", strip=True) or None


def extrair_precos(card):
    # Separa o bloco de preço em (preco, condominio, iptu), ainda como texto
    bloco = card.select_one('[data-cy="rp-cardProperty-price-txt"]')
    if bloco is None:
        return None, None, None
    paragrafos = bloco.find_all("p", recursive=False)
    preco = paragrafos[0].get_text(" ", strip=True) if paragrafos else None

    condominio = iptu = None
    if len(paragrafos) > 1:                  # "Cond. R$ 350 • IPTU isento"
        for parte in paragrafos[1].get_text(" ", strip=True).split("•"):
            parte = parte.strip()
            if parte.startswith("Cond."):
                condominio = parte.removeprefix("Cond.").strip()
            elif parte.startswith("IPTU"):
                iptu = parte.removeprefix("IPTU").strip()
    return preco, condominio, iptu


def extrair_bairro_anuncio(card):
    # "Asa Sul, Brasília" → "Asa Sul"; em lançamentos o bairro vem no campo da rua
    local = texto(card, "rp-cardProperty-location-txt") or ""
    rua = texto(card, "rp-cardProperty-street-txt") or ""
    for candidato in (local, rua):
        partes = [p.strip() for p in candidato.split(",")]
        if len(partes) >= 2 and partes[-1] == CIDADE_NOME:
            return partes[-2]
    return None


def extrair_id(link):
    # Lançamentos têm várias unidades com o mesmo "id-"; nelas o identificador é o development_unit_id
    achado = re.search(r"development_unit_id=(\d+)", link) or re.search(r"-id-(\d+)", link)
    return achado.group(1) if achado else None


def extrair_card(card, bairro, tipo):
    link_tag = card.select_one("a[href]")
    link = link_tag["href"] if link_tag else ""
    preco, condominio, iptu = extrair_precos(card)
    return {                                 # ← um dicionário = uma linha da tabela
        "id": extrair_id(link),
        "preco": preco,
        "condominio": condominio,
        "iptu": iptu,
        "area_m2": texto(card, "rp-cardProperty-propertyArea-txt"),
        "quartos": texto(card, "rp-cardProperty-bedroomQuantity-txt"),
        "banheiros": texto(card, "rp-cardProperty-bathroomQuantity-txt"),
        "vagas": texto(card, "rp-cardProperty-parkingSpacesQuantity-txt"),
        "bairro": bairro,
        "bairro_anuncio": extrair_bairro_anuncio(card),
        "tipo": tipo,
        "lancamento": card.get("data-type") == "FIXED DEVELOPMENTS POSITIONS",
        "url": link if link.startswith("http") else None,   # cards agrupados pelo site vêm com href="#"
    }

### Teste com uma página antes da coleta completa

In [7]:
sessao = criar_sessao()
url_teste = f"{CIDADE_URL}bairros/{BAIRROS['Asa Sul']}/{TIPOS['apartamento']}/"
sopa = baixar_pagina(sessao, url_teste, pagina=1)

cards = sopa.select(SELETOR_CARD)
print(f"URL: {url_teste}")
titulo_busca = sopa.select_one('[data-cy="rp-searchTitle-txt"]').get_text(strip=True)
print(f"Título da busca: {titulo_busca}")
print(f"Cards encontrados: {len(cards)}")

imovel = extrair_card(cards[0], "Asa Sul", "apartamento")
exibir_imovel(imovel)

URL: https://www.vivareal.com.br/venda/distrito-federal/brasilia/bairros/asa-sul/apartamento_residencial/
Título da busca: 127 Apartamentos para venda em Asa Sul, Brasília - DF
Cards encontrados: 30
╭───────────────────────────╮
│      RESUMO DO IMÓVEL     │
├───────────────────────────┤
│ Bairro (busca): Asa Sul   │
│ Bairro (anúncio): Asa Sul │
│ Tipo: apartamento         │
│ Preço: R$ 69.000          │
│ Condomínio: isento        │
│ IPTU: isento              │
│ Área: 57 m²               │
│ Quartos: 2                │
│ Banheiros: 2              │
│ Vagas: 1                  │
╰───────────────────────────╯


## Coleta completa

Percorre **bairro × tipo × página** até a página vir vazia, incompleta (menos de 30 cards = última página) ou chegar no teto `MAX_PAGINAS_POR_BUSCA`.

Se `imoveis_brutos.csv` já existir, **não raspa de novo** (regra de boa convivência com o site). Para forçar uma coleta nova, use `FORCAR_COLETA = True`.

In [8]:
def coletar():
    sessao = criar_sessao()
    imoveis = []
    for bairro, slug_bairro in BAIRROS.items():
        for tipo, slug_tipo in TIPOS.items():
            url_busca = f"{CIDADE_URL}bairros/{slug_bairro}/{slug_tipo}/"
            for pagina in range(1, MAX_PAGINAS_POR_BUSCA + 1):
                sopa = baixar_pagina(sessao, url_busca, pagina)
                pausa_educada()                            # pausa de 2 a 4 s: visitante, não ataque
                cards = sopa.select(SELETOR_CARD) if sopa else []
                status = "404 (não existe)" if sopa is None else f"{len(cards):>2} anúncios"
                print(f"{bairro:<14} | {tipo:<11} | página {pagina:>2} → {status}")

                imoveis.extend(extrair_card(card, bairro, tipo) for card in cards)
                if len(cards) < ANUNCIOS_POR_PAGINA:       # vazia ou incompleta = última página
                    break

    tabela = pd.DataFrame(imoveis)                         # lista de dicionários → tabela
    repetido = tabela["id"].notna() & tabela["id"].duplicated()
    print(f"\nColetados: {len(tabela)} | repetidos (mesmo id): {repetido.sum()}")
    tabela = tabela[~repetido].reset_index(drop=True)
    tabela["data_coleta"] = date.today().isoformat()
    return tabela


if FORCAR_COLETA or not ARQUIVO_BRUTO.exists():
    inicio = time.perf_counter()
    bruto = coletar()
    bruto.to_csv(ARQUIVO_BRUTO, index=False, encoding="utf-8-sig")   # utf-8-sig: Excel abre com acentos certos
    print(f"💾 {len(bruto)} imóveis salvos em {ARQUIVO_BRUTO} ({time.perf_counter() - inicio:.0f}s)")
else:
    print(f"♻️ Reaproveitando {ARQUIVO_BRUTO} (FORCAR_COLETA = False)")

bruto = pd.read_csv(ARQUIVO_BRUTO, dtype=str)            # tudo como texto cru, igual ao que veio do site
bruto["lancamento"] = bruto["lancamento"] == "True"
print(bruto.shape)

Asa Sul        | cobertura   | página  1 →  1 anúncios


Asa Sul        | kitnet      | página  1 →  4 anúncios


Asa Sul        | casa        | página  1 → 25 anúncios


Asa Sul        | apartamento | página  1 → 30 anúncios


Asa Sul        | apartamento | página  2 → 30 anúncios


...

Lago Norte     | kitnet      | página  1 →  5 anúncios


Lago Norte     | casa        | página  1 → 30 anúncios


Lago Norte     | casa        | página  2 → 30 anúncios


Lago Norte     | casa        | página  3 → 11 anúncios


Lago Norte     | apartamento | página  1 → 26 anúncios

Coletados: 798 | repetidos (mesmo id): 64
💾 734 imóveis salvos em imoveis_brutos.csv (180s)
(734, 14)


## Conferindo o resultado

In [9]:
bruto.head(10)

,id,preco,condominio,iptu,area_m2,quartos,banheiros,vagas,bairro,bairro_anuncio,tipo,lancamento,url,data_coleta
0,2821599338,R$ 1.959.000,isento,isento,125 m²,2,2,1,Asa Sul,Asa Sul,cobertura,False,https://www.vivareal.com.br/imovel/cobertura-2...,2026-09-27
1,2893493673,R$ 360.000,R$ 390,R$ 819,27 m²,1,1,2,Asa Sul,Asa Sul,kitnet,False,https://www.vivareal.com.br/imovel/kitnet-1-qu...,2026-09-27
2,2863998901,R$ 345.000,isento,isento,25 m²,1,1,1,Asa Sul,Asa Sul,kitnet,False,https://www.vivareal.com.br/imovel/kitnet-1-qu...,2026-09-27
3,2899174503,R$ 410.000,isento,isento,27 m²,1,1,NaN,Asa Sul,Asa Sul,kitnet,False,https://www.vivareal.com.br/imovel/kitnet-1-qu...,2026-09-27
4,2735690215,R$ 390.000,R$ 431,R$ 2.250,26 m²,1,1,2,Asa Sul,Asa Sul,kitnet,False,https://www.vivareal.com.br/imovel/kitnet-1-qu...,2026-09-27
5,2874672076,R$ 2.750.000,isento,R$ 3.600,330 m²,4,4,5,Asa Sul,Asa Sul,casa,False,https://www.vivareal.com.br/imovel/casa-4-quar...,2026-09-27
6,2893973476,R$ 1.650.000,isento,R$ 3.564,339 m²,4,6,2,Asa Sul,Asa Sul,casa,False,https://www.vivareal.com.br/imovel/casa-4-quar...,2026-09-27
7,2876887209,R$ 2.320.000,isento,isento,327 m²,5,7,3,Asa Sul,Asa Sul,casa,False,https://www.vivareal.com.br/imovel/casa-5-quar...,2026-09-27
8,2907628133,R$ 1.200.000,isento,isento,800 m²,3,5,NaN,Asa Sul,Asa Sul,casa,False,https://www.vivareal.com.br/imovel/casa-3-quar...,2026-09-27
9,2617010830,R$ 1.950.000,isento,isento,410 m²,4,6,8,Asa Sul,Asa Sul,casa,False,https://www.vivareal.com.br/imovel/casa-4-quar...,2026-09-27


In [10]:
print("Imóveis por bairro × tipo:")
display(pd.crosstab(bruto["bairro"], bruto["tipo"], margins=True, margins_name="Total"))

print("Valores ausentes (texto cru, antes da limpeza):")
print(bruto.isna().sum())

print(f"\nLançamentos ('A partir de'): {bruto['lancamento'].sum()}")
print(f"Coleta de: {bruto['data_coleta'].iloc[0]}")

Imóveis por bairro × tipo:


tipo,apartamento,casa,cobertura,kitnet,Total
bairro,,,,,
Asa Norte,197,11,11,23,242
Asa Sul,121,25,1,4,151
Cruzeiro Novo,14,0,0,0,14
Lago Norte,21,71,0,5,97
Lago Sul,0,46,0,1,47
Noroeste,75,1,9,0,85
Octogonal,3,0,0,0,3
Sudoeste,71,0,4,20,95
Total,502,154,25,53,734


Valores ausentes (texto cru, antes da limpeza):
id                 12
preco               0
condominio          4
iptu                4
area_m2             1
quartos             4
banheiros           1
vagas             185
bairro              0
bairro_anuncio      5
tipo                0
lancamento          0
url                12
data_coleta         0
dtype: int64

Lançamentos ('A partir de'): 3
Coleta de: 2026-09-27


In [11]:
# Um exemplo de cada tipo, só para bater o olho
for _, linha in bruto.groupby("tipo").head(1).iterrows():
    exibir_imovel(linha.to_dict(), titulo=f"EXEMPLO: {linha['tipo'].upper()}")

╭───────────────────────────╮
│     EXEMPLO: COBERTURA    │
├───────────────────────────┤
│ Bairro (busca): Asa Sul   │
│ Bairro (anúncio): Asa Sul │
│ Tipo: cobertura           │
│ Preço: R$ 1.959.000       │
│ Condomínio: isento        │
│ IPTU: isento              │
│ Área: 125 m²              │
│ Quartos: 2                │
│ Banheiros: 2              │
│ Vagas: 1                  │
╰───────────────────────────╯
╭───────────────────────────╮
│      EXEMPLO: KITNET      │
├───────────────────────────┤
│ Bairro (busca): Asa Sul   │
│ Bairro (anúncio): Asa Sul │
│ Tipo: kitnet              │
│ Preço: R$ 360.000         │
│ Condomínio: R$ 390        │
│ IPTU: R$ 819              │
│ Área: 27 m²               │
│ Quartos: 1                │
│ Banheiros: 1              │
│ Vagas: 2                  │
╰───────────────────────────╯
╭───────────────────────────╮
│       EXEMPLO: CASA       │
├───────────────────────────┤
│ Bairro (busca): Asa Sul   │
│ Bairro (anúncio): Asa Sul │
│ Tipo: ca